# Módulo 2 — Detección de latidos con Pan-Tompkins

En este cuaderno corremos nuestro detector (`pan_tompkins.py`) y frenamos después de cada etapa para ver qué le
hizo a la señal y por qué la necesitamos. La idea es que cualquiera del grupo pueda seguir el razonamiento sin
abrir el código, y que si algo falla se pueda ver en qué etapa se rompió.

Lo que hace el algoritmo, en orden:

1. **Pasabanda de 5 a 15 Hz**, para quedarnos con la banda donde está la energía del QRS.
2. **Derivada de cinco puntos**, para resaltar las pendientes empinadas del QRS.
3. **Cuadrado**, para que todo sea positivo y las pendientes grandes pesen mucho más que las chicas.
4. **Integración con una ventana móvil de 150 ms**, para juntar la energía de cada QRS en un solo pulso.
5. **Decisión**: un máximo de la señal integrada es un QRS si supera un umbral fijo (una fracción del máximo) y
   está a más de 200 ms del anterior.
6. **Ubicación del pico R** en la señal original, buscando el extremo en ±100 ms alrededor de cada QRS.

Las funciones se programan en `pan_tompkins.py`; acá sólo las llamamos, graficamos y discutimos.

In [ ]:
# Permite usar el código de las otras carpetas del repo
import sys
sys.path.insert(0, "../..")

# Recarga los cambios que hagas en los archivos .py sin reiniciar el cuaderno
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, freqz, welch

from modulos.m2_pan_tompkins import pan_tompkins as pt
from modulos.herramientas import graficos

graficos.aplicar_estilo()
CM = 1 / 2.54
COLOR_SENAL = graficos.COLOR_NEUTRO
COLOR_MARCA = graficos.COLOR_CONDICION["post_ejercicio"]
COLOR_REFERENCIA = graficos.COLOR_CONDICION["reposo"]

## 0. La señal de entrada

- `x`: la señal, como un array de una dimensión (la salida del módulo 1, aunque también lo hemos probado sobre la señal cruda, porque el propio pasabanda saca la continua y la deriva de la línea de base);
- `fs`: su frecuencia de muestreo en Hz;
- `nombre`: para los títulos de los gráficos

In [ ]:
x = ...                  # completar: la señal a analizar (array de una dimensión)
fs = ...                 # completar: frecuencia de muestreo [Hz]
nombre = ""              # completar: por ejemplo "V1 reposo"

DESDE, HASTA = 2, 8      # tramo (en segundos) que se muestra en los gráficos de cada etapa

x = np.asarray(x, dtype=float)
fs = float(fs)
t = np.arange(len(x)) / fs
print(f"{nombre}: {len(x)} muestras a {fs:g} Hz ({len(x) / fs / 60:.1f} min)")

In [ ]:
def en_tramo(desde=DESDE, hasta=HASTA):
    # Máscara booleana de las muestras entre desde y hasta (en segundos)
    return (t >= desde) & (t < hasta)


def indices_en_tramo(indices, desde=DESDE, hasta=HASTA):
    return indices[(indices >= desde * fs) & (indices < hasta * fs)]


def graficar_antes_y_despues(antes, despues, titulo_antes, titulo_despues, desde=DESDE, hasta=HASTA):
    # Dos señales una debajo de la otra, en el mismo tramo de tiempo
    m = en_tramo(desde, hasta)
    fig, ejes = plt.subplots(2, 1, sharex=True, figsize=(16 * CM, 8 * CM))
    for ax, senal, titulo in zip(ejes, (antes, despues), (titulo_antes, titulo_despues)):
        ax.plot(t[m], senal[m], color=COLOR_SENAL, lw=0.8)
        ax.set_title(titulo, loc="left")
    ejes[-1].set_xlabel("Tiempo [s]")
    fig.tight_layout()
    return fig, ejes

Antes de tocar nada, miramos la señal de entrada y su espectro. El espectro es lo que justifica el pasabanda de
la etapa 1: queremos ver dónde está la energía del ECG y dónde están los ruidos.

In [ ]:
f_welch, psd = welch(x, fs, nperseg=int(4 * fs))

fig, (ax_t, ax_f) = plt.subplots(2, 1, figsize=(16 * CM, 10 * CM))
m = en_tramo()
ax_t.plot(t[m], x[m], color=COLOR_SENAL, lw=0.8)
ax_t.set_title(f"Señal de entrada — {nombre}", loc="left")
ax_t.set_xlabel("Tiempo [s]")
ax_f.semilogy(f_welch, psd, color=COLOR_SENAL, lw=0.8)
ax_f.axvspan(*pt.BANDA_HZ, color=COLOR_REFERENCIA, alpha=0.15, label="Banda del pasabanda (5-15 Hz)")
ax_f.set_xlim(0, min(fs / 2, 80))
ax_f.set_xlabel("Frecuencia [Hz]")
ax_f.set_ylabel("Densidad espectral")
ax_f.legend(loc="upper right")
fig.tight_layout()

**Qué mirar:** casi toda la potencia está por debajo de unos pocos hertz. Ahí están la deriva de la línea de base
(respiración, movimiento) y las ondas P y T, que son lentas. El QRS es un evento rápido, de unos 80 a 120 ms, y por
eso su energía se reparte en frecuencias más altas. Si hay interferencia de línea, aparece como un pico angosto en
50 Hz.

Ahora corremos el algoritmo completo una sola vez y guardamos lo que sale de cada etapa, que es lo que vamos a ir
mirando de a una.

In [ ]:
etapas = pt.pan_tompkins(x, fs)

## 1. Pasabanda de 5 a 15 Hz

**Para qué está.** Pan y Tompkins observaron que la mayor parte de la energía del QRS cae entre 5 y 15 Hz. Si nos
quedamos con esa banda, por debajo sacamos la deriva de la línea de base y buena parte de las ondas P y T, y por
encima la interferencia de línea y el ruido muscular, que es de banda ancha. No buscamos que la señal "se vea
linda" (la forma del ECG cambia bastante), sino que el QRS se destaque frente a todo lo demás.

**Cómo lo hicimos y por qué distinto del artículo.** El artículo usa un pasabajos y un pasaaltos con coeficientes
enteros, pensados para correr en tiempo real en un microprocesador de los 80 a 200 Hz. Nosotros usamos un
Butterworth de orden 2 que se diseña para cualquier `fs`, y lo aplicamos con `filtfilt`: una vez hacia adelante y
otra hacia atrás. Así la fase total es cero y **la señal no se corre en el tiempo**, que es justamente lo que nos
pide el resto del trabajo (los picos R tienen que quedar en la muestra correcta). La contra de `filtfilt` es que
necesita la señal completa, pero nosotros analizamos registros ya grabados, así que no nos afecta.

Un detalle: al pasar dos veces por el filtro, la ganancia se eleva al cuadrado. En las frecuencias de corte, donde
un Butterworth tiene −3 dB, el conjunto queda en −6 dB, y fuera de la banda atenúa el doble (en dB).

In [ ]:
b, a = butter(pt.ORDEN_PASABANDA, pt.BANDA_HZ, btype="bandpass", fs=fs)
f_filtro, h = freqz(b, a, worN=4096, fs=fs)
ganancia_db = 20 * np.log10(np.abs(h) + 1e-12)

fig, ax = plt.subplots(figsize=(16 * CM, 6 * CM))
ax.plot(f_filtro, ganancia_db, color=COLOR_SENAL, ls="--", lw=0.8, label="Una pasada")
ax.plot(f_filtro, 2 * ganancia_db, color=COLOR_MARCA, lw=1, label="Ida y vuelta (filtfilt)")
ax.axhline(-3, color="k", lw=0.5, ls=":")
ax.axhline(-6, color="k", lw=0.5, ls=":")
ax.set_xlim(0, min(fs / 2, 80))
ax.set_ylim(-60, 3)
ax.set_xlabel("Frecuencia [Hz]")
ax.set_ylabel("Ganancia [dB]")
ax.set_title("Respuesta en frecuencia del pasabanda", loc="left")
ax.legend()
fig.tight_layout()

In [ ]:
fig, ejes = graficar_antes_y_despues(x, etapas["filtrada"], "Señal de entrada", "Después del pasabanda de 5-15 Hz")

**Qué mirar:** la línea de base queda plana y centrada en cero, las ondas P y T se achican mucho y el QRS queda como
una oscilación corta y bien marcada. Que la forma ya no se parezca a un ECG es esperable: la señal filtrada sólo se
usa para detectar, y el pico R después lo buscamos en la señal original (etapa 6).

Para comprobar que `filtfilt` no corrió la señal, superponemos un latido antes y después del filtro: los dos tienen
que estar alineados en el tiempo.

In [ ]:
# Tomamos un pico R detectado en el medio del tramo y miramos ±150 ms alrededor
r = indices_en_tramo(etapas["picos_r"])[len(indices_en_tramo(etapas["picos_r"])) // 2]
m = slice(r - int(0.15 * fs), r + int(0.15 * fs))


def normalizar(s):
    return s / np.max(np.abs(s))


fig, ax = plt.subplots(figsize=(10 * CM, 6 * CM))
ax.plot(1000 * (t[m] - t[r]), normalizar(x[m] - np.median(x[m])), color=COLOR_SENAL, label="Entrada")
ax.plot(1000 * (t[m] - t[r]), normalizar(etapas["filtrada"][m]), color=COLOR_MARCA, label="Filtrada")
ax.axvline(0, color="k", lw=0.5, ls=":")
ax.set_xlabel("Tiempo respecto del pico R [ms]")
ax.set_ylabel("Amplitud normalizada")
ax.legend()
fig.tight_layout()

## 2. Derivada de cinco puntos

**Para qué está.** Lo que distingue al QRS de las ondas P y T no es tanto su amplitud como lo **rápido** que
cambia: sube y baja en unas decenas de milisegundos. La derivada mide justamente eso, la pendiente, así que el QRS
queda grande y lo que cambia despacio queda chico.

**Cómo lo hicimos.** Usamos la misma derivada de cinco puntos del artículo,

$$y[n] = \frac{f_s}{8}\,\big(x[n+2] + 2x[n+1] - 2x[n-1] - x[n-2]\big),$$

pero centrada en la muestra `n` (usa dos muestras hacia adelante y dos hacia atrás). El artículo la usa causal, con
un retardo de dos muestras que después hay que compensar; como nosotros tenemos la señal entera, la centramos y nos
ahorramos esa corrección. El factor `fs/8` hace que el resultado esté en unidades por segundo.

Para ver que realmente se comporta como una derivada, comparamos su respuesta en frecuencia con la de una derivada
ideal, cuya ganancia crece proporcional a la frecuencia ($|H| = 2\pi f$).

In [ ]:
nucleo = np.array([1.0, 2.0, 0.0, -2.0, -1.0]) * fs / 8      # el mismo núcleo que usa pt.derivar
f_der, h_der = freqz(nucleo, 1, worN=4096, fs=fs)

fig, ax = plt.subplots(figsize=(16 * CM, 6 * CM))
ax.plot(f_der, np.abs(h_der), color=COLOR_MARCA, label="Derivada de cinco puntos")
ax.plot(f_der, 2 * np.pi * f_der, color=COLOR_SENAL, ls="--", lw=0.8, label="Derivada ideal")
ax.axvspan(*pt.BANDA_HZ, color=COLOR_REFERENCIA, alpha=0.15, label="Banda del pasabanda")
ax.set_xlim(0, fs / 2)
ax.set_ylim(0, 1.2 * np.abs(h_der).max())
ax.set_xlabel("Frecuencia [Hz]")
ax.set_ylabel("Ganancia [1/s]")
ax.set_title("Respuesta en frecuencia de la derivada", loc="left")
ax.legend()
fig.tight_layout()

**Qué mirar:** dentro de la banda que dejó pasar el filtro (5-15 Hz), las dos curvas casi coinciden, así que ahí es
una derivada de verdad. La de cinco puntos llega a un máximo en `fs/6` y después cae, o sea que además atenúa algo
del ruido de alta frecuencia; una derivada ideal haría lo contrario y lo amplificaría.

In [ ]:
fig, ejes = graficar_antes_y_despues(etapas["filtrada"], etapas["derivada"], "Filtrada", "Derivada")

**Qué mirar:** cada QRS se transforma en un par de lóbulos, uno positivo (donde la señal sube) y uno negativo (donde
baja). En el pico R la derivada pasa por cero, porque ahí la pendiente es nula. Lo que quedaba de las ondas P y T
casi desaparece, porque son lentas.

## 3. Cuadrado

**Para qué está.** Tiene dos funciones. La primera es que todo quede positivo: a la etapa siguiente no le importa si
la pendiente era de subida o de bajada, sólo cuánta pendiente hubo, y si sumáramos lóbulos con signo se
cancelarían. La segunda es que el cuadrado no es lineal: una pendiente el doble de grande queda cuatro veces más
grande. Eso agranda la diferencia entre el QRS, que tiene las pendientes más grandes, y todo lo demás (restos de la
onda T, ruido). Por eso usamos el cuadrado y no el valor absoluto, que también haría todo positivo pero no
acentuaría las diferencias.

In [ ]:
fig, ejes = graficar_antes_y_despues(etapas["derivada"], etapas["al_cuadrado"], "Derivada", "Derivada al cuadrado")

**Qué mirar:** quedan picos angostos y altos en cada QRS, casi en cero en el resto. Muchas veces cada QRS queda con
dos o más picos (uno por cada lóbulo de la derivada), con un valle en el medio. Si decidiéramos sobre esta señal,
podríamos contar un mismo latido dos veces; de eso se encarga la etapa siguiente.

## 4. Integración con ventana móvil de 150 ms

**Para qué está.** Reemplaza cada muestra por el promedio de la señal al cuadrado en una ventana de 150 ms centrada
en ella. Con eso los picos que tiene cada QRS se funden en un solo pulso, cuya altura indica cuánta energía de
pendiente hubo en ese intervalo. Esta es la señal sobre la que se toma la decisión.

**Por qué 150 ms.** El ancho de la ventana tiene que parecerse al ancho de un QRS, que normalmente está entre 80 y
120 ms. Si la ventana es muy corta, el QRS sigue partido en varios picos; si es muy larga, el pulso del QRS se
mezcla con la onda T y, a frecuencias cardíacas altas, con el latido siguiente. Los 150 ms del artículo (30 muestras
a 200 Hz) dan un poco de margen por encima de un QRS normal. Nosotros lo definimos en segundos, así sirve para
cualquier `fs`, y la ventana es centrada, de nuevo para no tener retardo.

In [ ]:
fig, ejes = graficar_antes_y_despues(etapas["al_cuadrado"], etapas["integrada"], "Derivada al cuadrado",
                                     f"Integrada (ventana de {1000 * pt.VENTANA_INTEGRACION_S:.0f} ms)")

Para ver el efecto del ancho de la ventana, repetimos la integración con otros anchos. Es la misma cuenta que hace
`pt.integrar_con_ventana_movil`, pero con la ventana como parámetro.

In [ ]:
anchos_s = [0.050, pt.VENTANA_INTEGRACION_S, 0.300]
m = en_tramo(DESDE, DESDE + 3)

fig, ejes = plt.subplots(len(anchos_s), 1, sharex=True, figsize=(16 * CM, 9 * CM))
for ax, ancho in zip(ejes, anchos_s):
    n = max(int(round(ancho * fs)), 1)
    integrada = np.convolve(etapas["al_cuadrado"], np.ones(n) / n, mode="same")
    ax.plot(t[m], integrada[m], color=COLOR_MARCA if ancho == pt.VENTANA_INTEGRACION_S else COLOR_SENAL, lw=0.8)
    ax.set_title(f"Ventana de {1000 * ancho:.0f} ms", loc="left")
    ax.set_yticks([])
ejes[-1].set_xlabel("Tiempo [s]")
fig.tight_layout()

**Qué mirar:** con 50 ms algunos QRS siguen con más de un pico, y con 300 ms los pulsos se ensanchan y aparece la
contribución de la onda T pegada al QRS. Con 150 ms cada latido queda como un solo pulso, que es lo que necesita la
etapa de decisión.

## 5. Decisión: umbral fijo y período refractario

**Qué hace.** Busca los máximos locales de la señal integrada y se queda con los que cumplen dos condiciones:

- **Superan el umbral**, que es una fracción fija (`UMBRAL_RELATIVO`, por defecto 0.3) del máximo de toda la señal
  integrada. Es una forma simple de que el umbral se ajuste a la amplitud de cada registro sin tener que elegir un
  número en mV para cada uno.
- **Están a más de 200 ms del QRS anterior** (período refractario). Después de despolarizarse, el ventrículo no
  puede volver a hacerlo hasta pasado un tiempo, así que dos latidos a menos de 200 ms (más de 300 lpm) no son
  fisiológicos. Si dos máximos quedan más cerca que eso, `find_peaks` se queda con el más alto. Esto evita, por
  ejemplo, contar como latido una onda T grande pegada al QRS.

**Por qué un umbral fijo y no los adaptativos del artículo.** Pan y Tompkins usan dos pares de umbrales que se
actualizan latido a latido según el nivel de "señal" y de "ruido", y una búsqueda hacia atrás cuando pasa mucho
tiempo sin detectar nada. Funciona muy bien, pero tiene muchos parámetros acoplados y es difícil entender por qué
detectó o dejó de detectar algo. Con un solo umbral fijo el detector es más simple y, sobre todo, podemos estudiar
directamente cómo cambia la detección al mover ese único parámetro, que es lo que hacemos más abajo.

**La contra** es que el umbral depende del máximo de todo el registro. Si hay un artefacto muy grande (un movimiento
brusco, un electrodo que se suelta), el máximo sube, el umbral sube con él y podemos perder latidos normales. Vale
la pena tenerlo en cuenta al revisar nuestros registros.

In [ ]:
m = en_tramo()
qrs = indices_en_tramo(etapas["qrs_integrada"])

fig, ax = plt.subplots(figsize=(16 * CM, 6 * CM))
ax.plot(t[m], etapas["integrada"][m], color=COLOR_SENAL, lw=0.8, label="Integrada")
ax.plot(t[m], etapas["umbral_integrada"][m], "--", color=COLOR_REFERENCIA, lw=0.8,
        label=f"Umbral ({pt.UMBRAL_RELATIVO:g} × máximo)")
ax.plot(t[qrs], etapas["integrada"][qrs], "v", ms=5, mfc="none", color=COLOR_MARCA, label="QRS detectado")
ax.set_xlabel("Tiempo [s]")
ax.set_title("Decisión sobre la señal integrada", loc="left")
ax.legend(loc="upper right", ncol=3)
fig.tight_layout()

**Cómo cambia la detección con el umbral.** Corremos el detector completo para varios valores de `umbral_relativo`.
Con un umbral muy bajo empiezan a pasar ondas T y ruido, y la cantidad de detecciones sube; con uno muy alto se
pierden los latidos más chicos, y baja. Lo que buscamos es una zona intermedia donde la cantidad casi no cambie (una
meseta): si el umbral que usamos cae ahí, la detección no depende mucho de haberlo elegido justo. Como referencia
graficamos también la cantidad de latidos que esperaríamos con la frecuencia cardíaca mediana.

In [ ]:
umbrales = np.round(np.arange(0.05, 0.91, 0.05), 2)
cantidades = [len(pt.detectar_picos_r(x, fs, umbral_relativo=u)) for u in umbrales]
rr_mediano = np.median(np.diff(etapas["picos_r"])) / fs
esperados = len(x) / fs / rr_mediano

fig, ax = plt.subplots(figsize=(16 * CM, 6 * CM))
ax.plot(umbrales, cantidades, "o-", ms=3, color=COLOR_MARCA, label="Latidos detectados")
ax.axhline(esperados, color=COLOR_REFERENCIA, lw=0.8, ls="--",
           label=f"Esperados con el RR mediano ({60 / rr_mediano:.0f} lpm)")
ax.axvline(pt.UMBRAL_RELATIVO, color="k", lw=0.5, ls=":", label="Umbral que usamos")
ax.set_xlabel("Umbral relativo (fracción del máximo de la integrada)")
ax.set_ylabel("Latidos detectados")
ax.legend()
fig.tight_layout()

for u, n in zip(umbrales, cantidades):
    print(f"umbral {u:.2f}: {n:5d} latidos detectados")

## 6. Ubicación del pico R

**Para qué está.** La etapa 5 nos dice dónde hay un QRS, pero sobre la señal integrada, que es un pulso ancho y
redondeado. Su máximo cae cerca del centro de la energía del QRS, no necesariamente en el pico R. Como para el
tacograma necesitamos el instante del R con la mayor precisión posible, volvemos a la señal de entrada y buscamos el
extremo dentro de una ventana de ±100 ms alrededor de cada QRS. Los 100 ms alcanzan para cubrir el corrimiento entre
el pulso integrado y el R, y son menos que la distancia típica entre el R y la onda T, así que no deberíamos
"saltar" a la T.

**La polaridad.** Si los electrodos quedaron al revés o la derivación tiene el QRS negativo, el R es un mínimo y no un
máximo. Para no depender de eso, antes de buscar los picos miramos en cada ventana cuánto sube la señal por encima de
su mediana y cuánto baja por debajo. Si en la mayoría de los latidos baja más de lo que sube, consideramos que el
registro está invertido y buscamos mínimos. La decisión se toma una vez para todo el registro (usando la mediana de
todos los latidos), así un latido raro no nos cambia el criterio.

Al final se ordenan los picos y se eliminan los repetidos, por si dos ventanas cercanas encontraron el mismo.

In [ ]:
desde, hasta = DESDE, DESDE + 3
m = en_tramo(desde, hasta)
semiancho = pt.SEMIANCHO_BUSQUEDA_S
qrs = indices_en_tramo(etapas["qrs_integrada"], desde, hasta)
picos = indices_en_tramo(etapas["picos_r"], desde, hasta)

fig, (ax_ecg, ax_int) = plt.subplots(2, 1, sharex=True, figsize=(16 * CM, 8 * CM))
ax_ecg.plot(t[m], x[m], color=COLOR_SENAL, lw=0.8)
ax_int.plot(t[m], etapas["integrada"][m], color=COLOR_SENAL, lw=0.8)
for i, q in enumerate(qrs):
    for ax in (ax_ecg, ax_int):
        ax.axvspan(t[q] - semiancho, t[q] + semiancho, color=COLOR_REFERENCIA, alpha=0.12,
                   label="Ventana de búsqueda (±100 ms)" if (i == 0 and ax is ax_ecg) else None)
        ax.axvline(t[q], color=COLOR_REFERENCIA, lw=0.6, ls="--")
ax_int.plot(t[qrs], etapas["integrada"][qrs], "v", ms=5, mfc="none", color=COLOR_MARCA, label="QRS (integrada)")
ax_ecg.plot(t[picos], x[picos], "o", ms=4, color=COLOR_MARCA, label="Pico R")
ax_ecg.set_title("Señal de entrada", loc="left")
ax_int.set_title("Integrada", loc="left")
ax_ecg.legend(loc="upper right", ncol=2)
ax_int.legend(loc="upper right")
ax_int.set_xlabel("Tiempo [s]")
fig.tight_layout()

corrimiento_ms = []
for p in etapas["picos_r"]:
    q = etapas["qrs_integrada"][np.argmin(np.abs(etapas["qrs_integrada"] - p))]
    corrimiento_ms.append(1000 * (p - q) / fs)
print(f"Distancia entre el QRS de la integrada y el pico R: mediana {np.median(corrimiento_ms):.0f} ms, "
      f"máxima {np.max(np.abs(corrimiento_ms)):.0f} ms (la ventana es de ±{1000 * semiancho:.0f} ms)")

**Prueba de la polaridad.** Si invertimos la señal, las etapas 1 a 5 dan exactamente lo mismo (los filtros son
lineales y el cuadrado borra el signo), y la etapa 6 debería darse cuenta de la inversión y encontrar los mismos
picos. Si esta celda imprime `True`, la detección de polaridad funciona.

In [ ]:
picos_normal = pt.detectar_picos_r(x, fs)
picos_invertida = pt.detectar_picos_r(-x, fs)
print("Mismos picos con la señal invertida:", np.array_equal(picos_normal, picos_invertida))

## Resultado final

Juntamos todo en una figura con las cinco señales intermedias una debajo de la otra. Es la que va al informe para
mostrar el paso a paso del algoritmo.

In [ ]:
def graficar_etapas(ecg, fs, etapas, desde=DESDE, hasta=HASTA, titulo=""):
    # Las etapas del algoritmo, una debajo de otra, en el tramo [desde, hasta) segundos
    t = np.arange(len(ecg)) / fs
    m = (t >= desde) & (t < hasta)

    def en_el_tramo(indices):
        return indices[(indices >= desde * fs) & (indices < hasta * fs)]

    picos = en_el_tramo(etapas["picos_r"])
    qrs = en_el_tramo(etapas["qrs_integrada"])
    filas = [
        (ecg, "1. Señal de entrada"),
        (etapas["filtrada"], "2. Pasabanda 5-15 Hz"),
        (etapas["derivada"], "3. Derivada"),
        (etapas["al_cuadrado"], "4. Cuadrado"),
        (etapas["integrada"], "5. Ventana móvil de 150 ms"),
    ]
    fig, ejes = plt.subplots(len(filas), 1, sharex=True, figsize=(16 * CM, 15 * CM))
    for ax, (senal, titulo_fila) in zip(ejes, filas):
        ax.plot(t[m], senal[m], color=COLOR_SENAL, lw=0.8)
        ax.set_title(titulo_fila, loc="left", fontsize=8, pad=2)
        ax.set_yticks([])
    ejes[0].plot(t[picos], ecg[picos], "o", ms=4, color=COLOR_MARCA, label="Pico R")
    ejes[-1].plot(t[m], etapas["umbral_integrada"][m], "--", lw=0.8, color=COLOR_REFERENCIA,
                  label="Umbral de detección")
    ejes[-1].plot(t[qrs], etapas["integrada"][qrs], "v", ms=4, mfc="none", color=COLOR_MARCA,
                  label="QRS detectado")
    ejes[-1].set_xlabel("Tiempo [s]")
    manejadores = [h for ax in (ejes[0], ejes[-1]) for h in ax.get_legend_handles_labels()[0]]
    fig.legend(handles=manejadores, loc="upper right", ncol=3, bbox_to_anchor=(1, 1))
    if titulo:
        fig.text(0.01, 0.985, titulo, va="top", fontsize=9)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    return fig


fig = graficar_etapas(x, fs, etapas, titulo=nombre)
# graficos.guardar_figura(fig, "pan_tompkins_etapas");   # descomentar para guardar la figura del informe

In [ ]:
picos = etapas["picos_r"]
rr = np.diff(picos) / fs
print(f"{len(picos)} latidos detectados en {len(x) / fs:.1f} s")
print(f"RR: mediana {1000 * np.median(rr):.0f} ms, mínimo {1000 * rr.min():.0f} ms, máximo {1000 * rr.max():.0f} ms")
print(f"Frecuencia cardíaca media: {60 / rr.mean():.1f} lpm")

## Revisión de las detecciones

Como no sabemos de antemano dónde están los latidos, la forma de encontrar errores es mirar los intervalos RR. Un
latido perdido aparece como un intervalo de casi el doble de lo normal, y una detección de más, como uno de casi la
mitad. La celda siguiente marca los intervalos que se apartan más de un 30 % de la mediana de sus vecinos. Después
hay que mirarlos con `ver()` para decidir si son errores del detector o latidos reales (por ejemplo, extrasístoles).

In [ ]:
def intervalos_para_revisar(picos, fs, tolerancia=0.3, vecinos=9):
    rr = np.diff(picos) / fs
    mediana = np.array([np.median(rr[max(i - vecinos // 2, 0): i + vecinos // 2 + 1]) for i in range(len(rr))])
    return np.where(np.abs(rr - mediana) > tolerancia * mediana)[0]


marcados = intervalos_para_revisar(picos, fs)
print(f"{len(marcados)} intervalos para revisar:",
      ", ".join(f"{picos[i + 1] / fs:.1f} s" for i in marcados[:15]))


def ver(desde, hasta):
    # Muestra un tramo de la señal con los picos detectados, para revisar a ojo
    m = en_tramo(desde, hasta)
    p = indices_en_tramo(picos, desde, hasta)
    fig, ax = plt.subplots(figsize=(16 * CM, 5 * CM))
    ax.plot(t[m], x[m], color=COLOR_SENAL, lw=0.8)
    ax.plot(t[p], x[p], "o", ms=4, mfc="none", mec=COLOR_MARCA)
    ax.set_xlabel("Tiempo [s]")
    fig.tight_layout()


# Ejemplo: ver(30, 40)

## Para seguir analizando

- [ ] Correr el cuaderno con cada uno de nuestros registros (cambiando la celda de la sección 0) y ver si la meseta del barrido de
      umbral está en el mismo lugar en reposo y post ejercicio.
- [ ] Revisar los intervalos marcados y anotar acá qué era cada uno.
- [ ] Fijarse si algún registro tiene artefactos grandes que suban el máximo de la integrada (y con él el umbral).
- [ ] Post ejercicio la frecuencia cardíaca es más alta: comprobar que la ventana de 150 ms y el refractario de
      200 ms siguen separando bien los latidos.